<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/02%20-%20Supervised%20Learning/solutions/02_exercises_solutions.ipynb)

# 02f · 第二模块综合习题参考答案（Solutions）

本文件提供 `02f_exercises.ipynb` 全部 8 道综合习题的完整可运行参考代码与结果核对。

In [1]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "02 - Supervised Learning/solutions"))
    get_ipython().run_line_magic("pip", "install -q rdkit")


## 0. 公共环境与 AqSolDB 数据准备

请先运行下方单元格加载清洗后的 9480 个分子及 17 个 RDKit 描述符：

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    root_mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
)

RDLogger.DisableLog("rdApp.*")

for p in ["data/solubility.csv", "../data/solubility.csv", "../01 - Basics/data/solubility.csv", "../../01 - Basics/data/solubility.csv"]:
    if os.path.exists(p):
        df = pd.read_csv(p)
        break

df["mol"] = df["SMILES"].apply(Chem.MolFromSmiles)
df = df[df["mol"].notna()].reset_index(drop=True)

feature_cols = [
    "MolWt", "MolLogP", "MolMR", "HeavyAtomCount", "NumHAcceptors", "NumHDonors",
    "NumHeteroatoms", "NumRotatableBonds", "NumValenceElectrons", "NumAromaticRings",
    "NumSaturatedRings", "NumAliphaticRings", "RingCount", "TPSA", "LabuteASA",
    "BalabanJ", "BertzCT",
]
X = df[feature_cols].values
y_reg = df["Solubility"].values
y_cls = (y_reg < -4.0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)
print(f"Loaded {len(df)} molecules | X_train: {X_train.shape} | X_test: {X_test.shape}")

Loaded 9480 molecules | X_train: (7584, 17) | X_test: (1896, 17)


---
## 第一部分：线性模型与正则化（对应 `02a`）

### 习题 1：均值猜测基线 vs 多元线性回归
1. 使用 `DummyRegressor(strategy="mean")` 在 `X_train, y_train` 上拟合并预测 `X_test`，计算测试集 RMSE；
2. 使用 `LinearRegression()` 在 `X_train, y_train` 上拟合并预测 `X_test`，打印测试集 RMSE 与 $R^2$（参考值：基线 RMSE $\approx 2.42$，OLS RMSE $\approx 1.64$）。

In [3]:
dummy_reg = DummyRegressor(strategy="mean").fit(X_train, y_train)
ols_reg = LinearRegression().fit(X_train, y_train)

pred_dummy = dummy_reg.predict(X_test)
pred_ols = ols_reg.predict(X_test)

print(f"Dummy Mean Baseline | Test RMSE = {root_mean_squared_error(y_test, pred_dummy):.3f}")
print(f"OLS Linear (17 desc)| Test RMSE = {root_mean_squared_error(y_test, pred_ols):.3f} | R² = {r2_score(y_test, pred_ols):.3f}")

Dummy Mean Baseline | Test RMSE = 2.418
OLS Linear (17 desc)| Test RMSE = 1.636 | R² = 0.542


### 习题 2：对比 OLS 与 Ridge 正则化对共线性权重膨胀的抑制
1. 先用只在 `X_train` 上 `fit` 的 `StandardScaler` 生成标准化矩阵 `X_train_s, X_test_s`；
2. 分别拟合 `LinearRegression()` 和 `Ridge(alpha=100.0)`，打印两者学到的 **最大权重绝对值 `np.max(np.abs(model.coef_))`**（观察共线特征导致的对冲大权重如何被 Ridge 瞬间压小！）。

In [4]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

ols_s = LinearRegression().fit(X_train_s, y_train)
ridge_s = Ridge(alpha=100.0).fit(X_train_s, y_train)

print(f"OLS 最大绝对权重   = {np.max(np.abs(ols_s.coef_)):.3f} (对应特征: {feature_cols[np.argmax(np.abs(ols_s.coef_))]})")
print(f"Ridge 最大绝对权重 = {np.max(np.abs(ridge_s.coef_)):.3f} (对应特征: {feature_cols[np.argmax(np.abs(ridge_s.coef_))]})")

OLS 最大绝对权重   = 6.326 (对应特征: HeavyAtomCount)
Ridge 最大绝对权重 = 1.402 (对应特征: BertzCT)


---
## 第二部分：距离与核边界模型（对应 `02b`）

### 习题 3：验证特征标准化对 k 近邻（kNN）与 RBF-SVR 的决定性影响
1. 对比 `KNeighborsRegressor(n_neighbors=6)` 在原始特征 `(X_train, X_test)` 与标准化特征 `(X_train_s, X_test_s)` 上的测试集 RMSE；
2. 在标准化特征 `(X_train_s, X_test_s)` 上训练 `SVR(kernel="rbf", C=5.0, epsilon=0.2)` 并打印测试集 RMSE（参考值：缩放后 kNN $\approx 1.23$，RBF-SVR $\approx 1.23$）。

In [5]:
knn_raw = KNeighborsRegressor(n_neighbors=6).fit(X_train, y_train)
knn_sc = KNeighborsRegressor(n_neighbors=6).fit(X_train_s, y_train)
svr_sc = SVR(kernel="rbf", C=5.0, epsilon=0.2).fit(X_train_s, y_train)

print(f"kNN (Raw)    Test RMSE = {root_mean_squared_error(y_test, knn_raw.predict(X_test)):.3f}")
print(f"kNN (Scaled) Test RMSE = {root_mean_squared_error(y_test, knn_sc.predict(X_test_s)):.3f}")
print(f"RBF-SVR      Test RMSE = {root_mean_squared_error(y_test, svr_sc.predict(X_test_s)):.3f}")

kNN (Raw)    Test RMSE = 1.554
kNN (Scaled) Test RMSE = 1.252


RBF-SVR      Test RMSE = 1.231


---
## 第三部分：决策树与集成学习（对应 `02c`）

### 习题 4：单棵深树的过拟合诊断 vs 随机森林 Bagging 降方差
分别在 `X_train, y_train` 上训练一棵不限深度的 `DecisionTreeRegressor(random_state=42)` 和包含 120 棵树的 `RandomForestRegressor(n_estimators=120, random_state=42, n_jobs=-1)`，打印两者各自的**训练集 RMSE** 与**测试集 RMSE**。

In [6]:
dt_deep = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)
rf_ens = RandomForestRegressor(n_estimators=120, random_state=42, n_jobs=-1).fit(X_train, y_train)

for name, m in [("Single Unlimited Tree", dt_deep), ("Random Forest (120)", rf_ens)]:
    tr_r = root_mean_squared_error(y_train, m.predict(X_train))
    te_r = root_mean_squared_error(y_test, m.predict(X_test))
    print(f"{name:22s} | Train RMSE = {tr_r:.3f} | Test RMSE = {te_r:.3f}")

Single Unlimited Tree  | Train RMSE = 0.056 | Test RMSE = 1.531
Random Forest (120)    | Train RMSE = 0.421 | Test RMSE = 1.169


---
## 第四部分：分类模型、概率与阈值移动（对应 `02d`）

### 习题 5：分层抽样与逻辑回归 vs 随机森林二分类评估
使用二分类标签 `y_cls`（$\log S < -4$ 为难溶正类 `1`）：
1. 用 `train_test_split(X, y_cls, test_size=0.2, stratify=y_cls, random_state=42)` 划分 `Xc_tr, Xc_te, yc_tr, yc_te`；
2. 分别拟合 `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` 和 `RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)`；
3. 打印两者在测试集上的 Accuracy、Precision、Recall、F1 与 ROC-AUC。

In [7]:
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X, y_cls, test_size=0.2, stratify=y_cls, random_state=42)

lr_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xc_tr, yc_tr)
rf_clf = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1).fit(Xc_tr, yc_tr)

for name, clf in [("Logistic Regression", lr_pipe), ("Random Forest Clf", rf_clf)]:
    pred = clf.predict(Xc_te)
    prob = clf.predict_proba(Xc_te)[:, 1]
    print(f"{name:20s} | Acc={accuracy_score(yc_te, pred):.3f} | Prec={precision_score(yc_te, pred):.3f} | "
          f"Rec={recall_score(yc_te, pred):.3f} | F1={f1_score(yc_te, pred):.3f} | AUC={roc_auc_score(yc_te, prob):.3f}")

Logistic Regression  | Acc=0.856 | Prec=0.833 | Rec=0.619 | F1=0.710 | AUC=0.892
Random Forest Clf    | Acc=0.890 | Prec=0.846 | Rec=0.752 | F1=0.796 | AUC=0.951


### 习题 6：按实验室容错需求下调报警概率阈值
取出上一题 `rf_clf` 对 `Xc_te` 预测的难溶概率 `probs = rf_clf.predict_proba(Xc_te)[:, 1]`，对比默认阈值 `0.50` 与保守拦截阈值 `0.25` 下的 **Recall（召回率）** 与 **Precision（精确率）**。

In [8]:
probs = rf_clf.predict_proba(Xc_te)[:, 1]
for th in [0.50, 0.25]:
    pred_th = (probs >= th).astype(int)
    print(f"Threshold = {th:.2f} | Recall = {recall_score(yc_te, pred_th):.3f} | Precision = {precision_score(yc_te, pred_th):.3f}")

Threshold = 0.50 | Recall = 0.758 | Precision = 0.845
Threshold = 0.25 | Recall = 0.889 | Precision = 0.733


---
## 第五部分：防泄漏交叉验证与骨架划分（对应 `02e`）

### 习题 7：使用 `make_pipeline` 执行无泄漏 5 折交叉验证
构建 `pipe = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=6, weights="distance"))`，在回归训练集 `X_train, y_train` 上用 `KFold(n_splits=5, shuffle=True, random_state=42)` 计算 5 折交叉验证 RMSE 的均值与标准差。

In [9]:
pipe_knn = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=6, weights="distance"))
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rmses = -cross_val_score(pipe_knn, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
print(f"5-Fold CV RMSE = {cv_rmses.mean():.3f} ± {cv_rmses.std():.3f}")

5-Fold CV RMSE = 1.202 ± 0.025


### 习题 8：实现 Murcko 骨架划分（Scaffold Split）检验跨家族泛化
1. 用 `MurckoScaffold.MurckoScaffoldSmiles(mol=m)` 提取 `df["mol"]` 中每个分子的核心骨架（空字符串替换为 `"acyclic"`）；
2. 用 `GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)` 按骨架分组切分 `(X, y_reg)`，训练 `RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)` 并打印全新骨架测试集上的 RMSE。

In [10]:
scaffolds = df["mol"].apply(lambda m: MurckoScaffold.MurckoScaffoldSmiles(mol=m)).replace("", "acyclic")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
s_tr, s_te = next(gss.split(X, y_reg, groups=scaffolds))

rf_scaf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X[s_tr], y_reg[s_tr])
print(f"Murcko Scaffold Split Test RMSE = {root_mean_squared_error(y_reg[s_te], rf_scaf.predict(X[s_te])):.3f}")

Murcko Scaffold Split Test RMSE = 1.193
